*Operadores Lógicos:*

Operadores lógicos são essenciais pra criar regras de decisão. Em análise de malware, isso aparece quando quero combinar sinais: score alto, API suspeita, tamanho estranho, seção executável, etc.

In [1]:
score_modelo = 0.87
tem_api_suspeita = True
tamanho_estranho = True

score_modelo > 0.80 and tem_api_suspeita

True

In [2]:
score_modelo = 0.62
tem_api_suspeita = True
assinatura_conhecida = False

tem_api_suspeita or assinatura_conhecida

True

In [3]:
arquivo_confiavel = False
not arquivo_confiavel

True

`and` exige que as duas condições sejam verdadeiras. `or` aceita que pelo menos uma seja verdadeira. `not` inverte o valor lógico.

In [4]:
familia = "ransomware"
score = 0.91
quantidade_imports = 180

familia == "ransomware" and score >= 0.90 and quantidade_imports > 100

True

In [5]:
extensao = ".exe"
extensao in [".exe", ".dll", ".sys"]

True

In [6]:
api = "CreateRemoteThread"

"Thread" in api, "Socket" not in api

(True, True)

Comparações encadeadas deixam a validação mais legível:

In [7]:
score = 0.76
0.70 <= score <= 0.90

True

`all()` e `any()` são úteis quando tenho várias verificações de uma amostra:

In [8]:
checks = [
    score_modelo > 0.50,
    tem_api_suspeita,
    quantidade_imports > 50
]

all(checks), any(checks)

(True, True)

In [9]:
amostra = {
    "hash": "a3d5f6e8b2c1",
    "familia": "trojan",
    "score": 0.88,
    "imports_suspeitos": ["VirtualAlloc", "WriteProcessMemory"]
}

if amostra["score"] >= 0.80 and len(amostra["imports_suspeitos"]) > 0:
    print("Amostra deve ser revisada com prioridade")
else:
    print("Amostra pode ficar na fila normal")

Amostra deve ser revisada com prioridade


*Loops:*

Loops servem para processar várias amostras, várias strings, várias APIs ou várias linhas de um dataset sem repetir código manualmente.

In [10]:
hashes = [
    "a3d5f6e8b2c1",
    "ff12aa98bb77",
    "001122334455"
]

for h in hashes:
    print(f"Processando hash: {h}")

Processando hash: a3d5f6e8b2c1
Processando hash: ff12aa98bb77
Processando hash: 001122334455


In [11]:
apis_importadas = ["CreateFileA", "VirtualAlloc", "WriteProcessMemory", "CloseHandle"]

for api in apis_importadas:
    if "Process" in api:
        print(f"API relacionada a processo: {api}")

API relacionada a processo: WriteProcessMemory


`enumerate()` é útil quando quero saber também a posição de cada item:

In [12]:
familias = ["trojan", "ransomware", "backdoor"]

for indice, familia in enumerate(familias):
    print(f"{indice} -> {familia}")

0 -> trojan
1 -> ransomware
2 -> backdoor


Loops com dicionários aparecem bastante quando estou analisando metadados:

In [13]:
metadados = {
    "hash_md5": "a3d5f6e8b2c1",
    "tipo": "PE",
    "familia": "WannaCry",
    "score": 0.94
}

for chave, valor in metadados.items():
    print(f"{chave}: {valor}")

hash_md5: a3d5f6e8b2c1
tipo: PE
familia: WannaCry
score: 0.94


`break` interrompe o loop. Isso é útil quando já encontrei o que procurava:

In [14]:
apis = ["CreateFileA", "ReadFile", "VirtualAlloc", "CloseHandle"]

for api in apis:
    if api == "VirtualAlloc":
        print("API suspeita encontrada")
        break

API suspeita encontrada


`continue` pula para a próxima repetição. Isso é útil para ignorar amostras inválidas:

In [15]:
amostras = ["sample1.exe", "", "sample2.dll", None, "sample3.sys"]

for amostra in amostras:
    if not amostra:
        continue
    print(f"Amostra válida: {amostra}")

Amostra válida: sample1.exe
Amostra válida: sample2.dll
Amostra válida: sample3.sys


`while` pode ser usado quando a repetição depende de uma condição, mas no projeto é melhor tomar cuidado pra não criar loop infinito:

In [16]:
tentativas = 0
max_tentativas = 3

while tentativas < max_tentativas:
    print(f"Tentativa {tentativas + 1}")
    tentativas += 1

Tentativa 1
Tentativa 2
Tentativa 3


Loop aninhado pode representar processamento de matrizes, como uma imagem gerada a partir dos bytes de um binário:

In [17]:
imagem_bytes = [
    [77, 90, 144],
    [0, 3, 0],
    [255, 128, 64]
]

for linha in imagem_bytes:
    for pixel in linha:
        print(pixel, end=" ")
    print()

77 90 144 
0 3 0 
255 128 64 


*Functions:*

Funções servem para transformar blocos repetidos em unidades reutilizáveis. No TCC, isso vai ser importante para limpar dados, extrair features, validar amostras e avaliar resultados.

In [18]:
def normalizar_hash(hash_texto):
    return hash_texto.strip().lower()

normalizar_hash("  A3D5F6E8B2C1  ")

'a3d5f6e8b2c1'

Uma função deve ter uma responsabilidade clara. Por exemplo: verificar se uma API é suspeita.

In [19]:
def api_suspeita(api):
    apis_perigosas = {"VirtualAlloc", "WriteProcessMemory", "CreateRemoteThread"}
    return api in apis_perigosas

api_suspeita("CreateRemoteThread"), api_suspeita("CloseHandle")

(True, False)

Funções também podem receber mais de um parâmetro:

In [20]:
def classificar_por_score(score, limite):
    if score >= limite:
        return "suspeita"
    return "normal"

classificar_por_score(0.87, 0.80)

'suspeita'

Funções podem retornar múltiplos valores usando tuplas:

In [21]:
def resumir_amostra(nome, score):
    prioridade = score >= 0.80
    return nome, score, prioridade

nome, score, prioridade = resumir_amostra("sample1.exe", 0.91)
nome, score, prioridade

('sample1.exe', 0.91, True)

*Docstrings:*

Docstring é um texto dentro da função explicando o que ela faz. Isso é importante porque o código do TCC precisa ser fácil de revisar, explicar e manter.

In [22]:
def extrair_extensao(nome_arquivo):
    """
    Recebe o nome de um arquivo e retorna a extensão em minúsculo.

    Exemplo:
    extrair_extensao("malware.EXE") -> ".exe"
    """
    return "." + nome_arquivo.split(".")[-1].lower()

extrair_extensao("malware.EXE")

'.exe'

In [23]:
help(extrair_extensao)

Help on function extrair_extensao in module __main__:

extrair_extensao(nome_arquivo)
    Recebe o nome de um arquivo e retorna a extensão em minúsculo.

    Exemplo:
    extrair_extensao("malware.EXE") -> ".exe"



Docstring boa não é enfeite (difícil de acreditar). Ela deve responder: o que entra, o que sai e qual é a regra principal da função.

In [ ]:
def calcular_prioridade(score, qtd_apis_suspeitas):
    """
    Calcula uma prioridade simples para revisão manual.

    Parâmetros:
        score: confiança do modelo entre 0 e 1.
        qtd_apis_suspeitas: quantidade de APIs suspeitas encontradas.

    Retorno:
        String indicando baixa, média ou alta prioridade.
    """
    if score >= 0.90 and qtd_apis_suspeitas >= 2:
        return "alta"
    if score >= 0.70 or qtd_apis_suspeitas >= 1:
        return "media"
    return "baixa"

calcular_prioridade(0.93, 3)

'alta'

*Map e Filter Functions:*

`map()` aplica uma função em todos os itens. `filter()` mantém apenas os itens que passam em uma condição.

In [25]:
hashes = [" A3D5F6E8B2C1 ", " FF12AA98BB77 ", " 001122334455 "]

hashes_normalizados = list(map(normalizar_hash, hashes))
hashes_normalizados

['a3d5f6e8b2c1', 'ff12aa98bb77', '001122334455']

Na prática, muitas vezes list comprehension é mais legível, mas `map()` ajuda a entender programação funcional:

In [26]:
hashes_normalizados = [normalizar_hash(h) for h in hashes]
hashes_normalizados

['a3d5f6e8b2c1', 'ff12aa98bb77', '001122334455']

In [27]:
apis = ["CreateFileA", "VirtualAlloc", "CloseHandle", "CreateRemoteThread"]

apis_filtradas = list(filter(api_suspeita, apis))
apis_filtradas

['VirtualAlloc', 'CreateRemoteThread']

Também posso usar `filter()` para remover dados vazios antes de processar:

In [28]:
nomes_arquivos = ["sample1.exe", "", None, "sample2.dll", "   ", "driver.sys"]

nomes_validos = list(filter(lambda nome: nome and nome.strip(), nomes_arquivos))
nomes_validos

['sample1.exe', 'sample2.dll', 'driver.sys']

*Lambda Expressions:*

Lambda é uma função pequena e anônima. É útil quando a regra é simples e usada apenas uma vez.

In [29]:
dobrar = lambda x: x * 2
dobrar(5)

10

In [30]:
scores = [0.91, 0.45, 0.78, 0.99, 0.62]

scores_altos = list(filter(lambda s: s >= 0.80, scores))
scores_altos

[0.91, 0.99]

In [31]:
familias_scores = [
    ("trojan", 0.83),
    ("ransomware", 0.97),
    ("backdoor", 0.74)
]

ordenado_por_score = sorted(familias_scores, key=lambda item: item[1], reverse=True)
ordenado_por_score

[('ransomware', 0.97), ('trojan', 0.83), ('backdoor', 0.74)]

Lambda é boa para regra curta. Se a lógica crescer, é melhor criar uma função normal com nome e docstring.

In [32]:
amostras = [
    {"nome": "sample1.exe", "score": 0.91},
    {"nome": "sample2.dll", "score": 0.54},
    {"nome": "sample3.sys", "score": 0.82}
]

nomes_suspeitos = list(map(lambda a: a["nome"], filter(lambda a: a["score"] >= 0.80, amostras)))
nomes_suspeitos

['sample1.exe', 'sample3.sys']

A mesma lógica pode ficar mais clara com list comprehension:

In [33]:
nomes_suspeitos = [a["nome"] for a in amostras if a["score"] >= 0.80]
nomes_suspeitos

['sample1.exe', 'sample3.sys']

*Métodos de Strings:*

Métodos de strings são muito usados para limpar nomes de arquivos, normalizar hashes, quebrar caminhos, procurar APIs e preparar dados textuais.

In [34]:
nome = "  Malware_Sample.EXE  "

nome.strip(), nome.lower(), nome.upper()

('Malware_Sample.EXE', '  malware_sample.exe  ', '  MALWARE_SAMPLE.EXE  ')

In [35]:
caminho = "C:/Users/analista/Desktop/sample.exe"
caminho.split("/")

['C:', 'Users', 'analista', 'Desktop', 'sample.exe']

In [36]:
nome_arquivo = caminho.split("/")[-1]
nome_arquivo

'sample.exe'

In [37]:
nome_arquivo.startswith("sample"), nome_arquivo.endswith(".exe")

(True, True)

In [38]:
api = "CreateRemoteThread"

api.lower(), api.find("Remote"), api.count("e")

('createremotethread', 6, 5)

In [39]:
assinatura = "CreateRemoteThread|VirtualAlloc|WriteProcessMemory"

assinatura.split("|")

['CreateRemoteThread', 'VirtualAlloc', 'WriteProcessMemory']

In [40]:
apis = ["CreateRemoteThread", "VirtualAlloc", "WriteProcessMemory"]

"|".join(apis)

'CreateRemoteThread|VirtualAlloc|WriteProcessMemory'

In [41]:
texto = "familia:WannaCry;score:0.94;tipo:PE"

texto.replace(";", " | ")

'familia:WannaCry | score:0.94 | tipo:PE'

`replace()` pode ser útil para padronizar strings antes de salvar resultados:

In [42]:
familia = "Wanna Cry"
familia_padronizada = familia.lower().replace(" ", "_")
familia_padronizada

'wanna_cry'

Strings também ajudam na criação de mensagens de log:

In [43]:
hash_md5 = "a3d5f6e8b2c1"
score = 0.94231
familia = "ransomware"

print(f"Amostra {hash_md5} classificada como {familia} com score {score:.2f}")

Amostra a3d5f6e8b2c1 classificada como ransomware com score 0.94


*Exemplo Integrado:*

Aqui junto operadores lógicos, loops, funções, lambda, filter e métodos de string em um fluxo pequeno parecido com o que pode aparecer no projeto.

In [44]:
def limpar_nome_arquivo(nome):
    """
    Normaliza o nome de um arquivo.

    Remove espaços, transforma em minúsculo e troca espaços internos por underline.
    """
    return nome.strip().lower().replace(" ", "_")

def deve_revisar(amostra):
    """
    Decide se uma amostra deve ir para revisão manual.

    A regra considera score alto ou presença de APIs suspeitas.
    """
    return amostra["score"] >= 0.80 or len(amostra["apis_suspeitas"]) > 0

amostras = [
    {
        "nome": " Sample 1.EXE ",
        "score": 0.91,
        "apis": ["CreateFileA", "VirtualAlloc"]
    },
    {
        "nome": " benign_tool.exe ",
        "score": 0.31,
        "apis": ["CreateFileA", "CloseHandle"]
    },
    {
        "nome": " loader.DLL ",
        "score": 0.74,
        "apis": ["CreateRemoteThread"]
    }
]

apis_perigosas = {"VirtualAlloc", "WriteProcessMemory", "CreateRemoteThread"}

for amostra in amostras:
    amostra["nome"] = limpar_nome_arquivo(amostra["nome"])
    amostra["apis_suspeitas"] = list(filter(lambda api: api in apis_perigosas, amostra["apis"]))

revisao = list(filter(deve_revisar, amostras))
revisao

[{'nome': 'sample_1.exe',
  'score': 0.91,
  'apis': ['CreateFileA', 'VirtualAlloc'],
  'apis_suspeitas': ['VirtualAlloc']},
 {'nome': 'loader.dll',
  'score': 0.74,
  'apis': ['CreateRemoteThread'],
  'apis_suspeitas': ['CreateRemoteThread']}]